In [ ]:
import sys
import numpy as np
import importlib
import dragons_chime_conv_combine
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u
from reproject import reproject_interp

In [ ]:
chime = {}

hdu = fits.open('/srv/data/chime/chime_QU_Oct2023_400_729/I_400_729_Oct2023_new.fits')
#hdu = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/DRAGONS_500_to_1024_i_cube.fits')
chime['i'] = hdu[0].data

hdu = fits.open('/srv/data/chime/chime_QU_Oct2023_400_729/Q_400_729_Oct2023_new_eq.fits')
#hdu = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/DRAGONS_500_to_1024_q_cube.fits')
chime['q'] = hdu[0].data

hdu = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2024_new.fits')
#hdu = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/DRAGONS_500_to_1024_u_cube.fits')
chime['u'] = hdu[0].data

hdu = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/V_400_729_Mar2024_new.fits')
chime['v'] = hdu[0].data

hdr = hdu[0].header
wcs = WCS(hdr)

freqs = wcs.all_pix2world(0,0,range(wcs.array_shape[0]),0)[2]/1e6
#freqs = np.genfromtxt('/srv/data/dragons/Mar2024_500_1000_MHz_cube/ListOfFrequencies_Hz.txt')/1e6

#print(freqs)
print(freqs.shape)
print(chime['q'].shape)

In [ ]:
def four_panel_fig(data, wcs, 
                   fidx=0, vmin=[0,-2,-2,-1], vmax=[5,2,2,1], 
                   maps=False, cmap=['RdBu_r','RdBu_r','RdBu_r','RdBu_r'],
                   RA_range=[24,0], dec_range=[-90,90],hfrac=1, uv_m=None,uv_n=None):

    fig = plt.figure(figsize=(12,12*hfrac))

    if maps:
        axi = fig.add_subplot(221, projection=wcs.celestial)
        axq = fig.add_subplot(222, projection=wcs.celestial)
        axu = fig.add_subplot(223, projection=wcs.celestial)
        axv = fig.add_subplot(224, projection=wcs.celestial)
        crange = SkyCoord(RA_range, dec_range, frame=ICRS, unit=(u.hourangle, u.deg))
    else:
        axi = fig.add_subplot(221)
        axq = fig.add_subplot(222)
        axu = fig.add_subplot(223)
        axv = fig.add_subplot(224)

    stokes = list(data.keys())
    axs    = [axi, axq, axu, axv]

    for i in range(0,4):

        if maps:
            axs[i].imshow(data[stokes[i]][fidx], vmin=vmin[i], vmax=vmax[i], origin='lower',cmap=cmap[i])
        else:
            #axs[i].imshow(abs(data[stokes[i]]), vmin=vmin[i], vmax=vmax[i], origin='lower',cmap=cmap[i],
            #             extent=(uv_m[0],uv_m[-1],uv_n[0],uv_n[-1]))
            #axs[i].set_xlim(uv_m[0],uv_m[-1])
            #axs[i].set_ylim(uv_n[0],uv_n[-1])
            axs[i].imshow(abs(data[stokes[i]]), vmin=vmin[i], vmax=vmax[i], origin='lower',cmap=cmap[i],
                         extent=(uv_m[0],uv_m[-1],uv_n[0],uv_n[-1]))
            axs[i].set_xlim(-120,120)
            axs[i].set_ylim(-120,120)
            axs[i].axvline(x=0,color='red')
            axs[i].axvline(x=22,color='red')
            axs[i].axvline(x=44,color='red')
            axs[i].axvline(x=66,color='red')

            #axs[i].set_xlim(-20,20)
            #axs[i].set_ylim(-20,20)

            
            axs[i].set_xlabel(r'$u$ (m)')
            axs[i].set_ylabel(r'$v$ (m)')
            axs[i].set_title('Stokes '+stokes[i].upper())

        if maps:
            axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
            axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        
    
    return

In [ ]:
fidx = np.argmin(abs(freqs-610))
print(fidx)

print(freqs[fidx])

In [ ]:
#four_panel_fig(chime, wcs.dropaxis(2), maps=False, fidx=fidx, RA_range=[24,0],dec_range=[0,90],hfrac=0.4,
#              cmap=['cubehelix','RdBu_r','RdBu_r','RdBu_r'], vmin=[0,-10,-10,-0.5], vmax=[50,10,10,0.5])

In [ ]:
plt.hist(chime['q'][fidx].flatten(),bins=101,range=(-1,1),alpha=0.5);
plt.hist(chime['u'][fidx].flatten(),bins=101,range=(-1,1),alpha=0.5);
plt.ylim(0,1e5)
plt.xlim(-1,1)
plt.axvline(x=0,color='k')

qmed = np.nanmedian(chime['q'][fidx][abs(chime['q'][fidx])>0])
umed = np.nanmedian(chime['u'][fidx][abs(chime['u'][fidx])>0]) 

print(qmed)
print(umed)

plt.axvline(x=qmed,color='blue')
plt.axvline(x=umed,color='red')

In [ ]:
#importlib.reload(dragons_chime_conv_combine)

chime_uv = {}

dxy = wcs.to_header()['CDELT2']
print(dxy)

for key in chime.keys():
    print(key)
    map_in = chime[key][fidx]
    map_in[np.isnan(map_in)] = 0
    image_FFT = np.fft.fft2(map_in)
    image_FFT_shift = np.fft.fftshift(image_FFT)
    chime_uv[key] = image_FFT_shift

uv_freq_m = np.fft.fftshift(np.fft.fftfreq(chime_uv['i'].shape[1]))/dxy
uv_freq_n = np.fft.fftshift(np.fft.fftfreq(chime_uv['i'].shape[0]))/dxy

uv_m = 4*uv_freq_m*(3e8/(freqs[fidx]*1e6))*180/np.pi**2
uv_n = 4*uv_freq_n*(3e8/(freqs[fidx]*1e6))*180/np.pi**2


print(uv_m)
print(uv_n)

In [ ]:
four_panel_fig(chime_uv, wcs.dropaxis(2), maps=False, fidx=fidx, RA_range=[24,0],dec_range=[0,90],hfrac=1,
              cmap=['viridis','viridis','viridis','viridis'], vmin=[0,0,0,0], vmax=[60000,5000,5000,5000],
              uv_m=uv_m,uv_n=uv_n)

#plt.savefig('/srv/data/gmims/gmims-lbn/experiments/dragons_uv_610MHz.png')

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(20,12))

uvn_pick = 10
uvn_idx = abs(uvn_pick-uv_n).argmin()
print(uv_n[uvn_idx])

ax[0].plot(uv_m, abs(chime_uv['i'][uvn_idx]),color='red')
ax[1].plot(uv_m, abs(chime_uv['q'][uvn_idx]))
ax[1].plot(uv_m, abs(chime_uv['u'][uvn_idx]))
ax[1].plot(uv_m, abs(chime_uv['v'][uvn_idx]))

ax[0].set_ylim(0,2e5)
ax[1].set_ylim(0,2e4)

for i in range(0,2):
    ax[i].set_xlim(-150,150)
    ax[i].axvline(x=0,color='black')
    ax[i].axvline(x=22,color='black')
    ax[i].axvline(x=44,color='black')
    ax[i].axvline(x=66,color='black')
    #ax[i].set_yscale('log')